# 01 - Day 1: INDB tables + Khana-to-INDB mapping
Run the cells top to bottom. **Do not save this notebook back to GitHub** (its outputs contain INDB rows).
All results are saved to Google Drive: `MyDrive/AaharAI/data/`.

In [ ]:
GITHUB_USER = "OnYourLeftover"

from google.colab import drive
drive.mount("/content/drive")
!pip -q install rapidfuzz
import sys
!git -C /content/AaharAI pull || git clone https://github.com/{GITHUB_USER}/AaharAI.git /content/AaharAI
sys.path.insert(0, "/content/AaharAI")
%load_ext autoreload
%autoreload 2

## Step 1: put INDB on Drive (once) and build the tables

In [ ]:
from pathlib import Path
DATA = Path("/content/drive/MyDrive/AaharAI/data"); DATA.mkdir(parents=True, exist_ok=True)
!test -f {DATA}/indb/INDB.xlsx || git clone https://github.com/lindsayjaacks/Indian-Nutrient-Databank-INDB.git {DATA}/indb
print("INDB files:", sorted(p.name for p in (DATA / "indb").glob("*.xlsx")))

In [ ]:
from src import indb
rec, kept, nutrition = indb.build_tables()
print("ingredient rows:", len(rec), "| kept recipes:", len(kept), "| nutrition rows:", len(nutrition))
print(rec.cat.value_counts().to_string())
print("duplicate recipe codes in kept:", kept.recipe_code.duplicated().sum())
print("guessed serving units:", int(nutrition.serving_guessed.sum()), "| no serving info:", int(nutrition.no_of_servings.isna().sum()), "| implausible values flagged:", int((nutrition.nutrition_flag == "check").sum()))
print("\nMost common unclassified ('other') ingredients:")
print(rec[rec.cat == "other"].canonical.value_counts().head(25).to_string())

In [ ]:
# sanity check: per-serving values should look like one serving of the stated unit
print(nutrition.sample(5, random_state=0)[["recipe_name", "serving_unit", "size_of_servings", "kcal_100g", "kcal", "serving_note"]].to_string(index=False))

## Step 2: Khana-to-INDB mapping (statuses: match, variant, proxy, none)
match = same dish. variant = same dish, recipe details differ. proxy = closest dish in the same family (shown as 'approximate'). none = no nutrition shown.
Re-running never overwrites rows you have already edited by hand.

In [ ]:
import pandas as pd
from src.khana_classes import KHANA_CLASSES

M = {  # label: (indb_recipe_id, status, note)
 "aloo gobi": ("ASC171", "match", "Potato cauliflower (Aloo gobhi)"),
 "aloo methi": ("ASC175", "match", ""),
 "aloo mutter": ("ASC190", "match", "Pea potato curry (Aloo matar)"),
 "aloo paratha": ("ASC098", "match", ""),
 "anda curry": ("BFP240", "match", "Egg curry (Anda curry)"),
 "banana chips": ("OSR119", "match", ""),
 "besan laddu": ("ASC343", "match", "Gram flour ladoo"),
 "chapati": ("ASC096", "match", "Chapati/Roti"),
 "chikki": ("ASC382", "match", "Peanut brittle; alt BFP413 murmura chikki"),
 "chivda": ("ASC052", "match", "Rice flakes (Chiwda/Aval); verify ingredients"),
 "dal khichdi": ("BFP144", "match", "Plain khichdi; alt BFP554"),
 "dhokla": ("ASC474", "match", "alts BFP440, BFP573, OSR115"),
 "fish curry": ("ASC246", "match", "alt BFP223 Bengal fish curry"),
 "gajar ka halwa": ("ASC295", "match", ""),
 "gujhia": ("ASC280", "match", "INDB spells it Gunjia"),
 "gulab jamun": ("ASC348", "match", "with khoya; alt BFP386"),
 "idli": ("ASC144", "match", "alts BFP147, BFP579"),
 "kaju katli": ("ASC339", "match", "Cashewnut burfi (Kaju katli)"),
 "khakhra": ("OSR102", "match", ""),
 "kheer": ("ASC282", "match", "rice kheer"),
 "kulfi": ("ASC321", "match", ""),
 "masala dosa": ("ASC146", "match", ""),
 "medu vada": ("BFP436", "match", "plain urad dal vada"),
 "moong dal halwa": ("ASC299", "match", ""),
 "murukku": ("OSR113", "match", "rice murukku"),
 "onion pakoda": ("ASC352", "match", ""),
 "palak paneer": ("ASC215", "match", ""),
 "pav bhaji": ("OSR112", "match", ""),
 "phirni": ("ASC292", "match", ""),
 "poha": ("BFP044", "match", "alt BFP045 vegetable poha"),
 "rasgulla": ("BFP392", "match", ""),
 "rava dosa": ("ASC147", "match", "semolina dosa"),
 "sabudana khichdi": ("OSR099", "match", ""),
 "sabudana vada": ("ASC379", "match", "peanut sago vada"),
 "uttapam": ("BFP152", "match", "alt ASC148"),
 "garlic naan": ("ASC142", "variant", "plain naan, garlic not in recipe"),
 "samosa": ("ASC361", "variant", "potato samosa"),
 "chana masala": ("ASC162", "variant", "chickpea curry; alt BFP185 kabuli channa curry"),
 "margherita pizza": ("ASC376", "variant", "cheese pizza; compare ingredients with BFP122 Pizza"),
 "bhindi masala": ("BFP269", "proxy", "okra fry (bhindi sabzi)"),
 "paneer masala": ("ASC195", "proxy", "generic paneer curry; alts ASC222 butter sauce, ASC221 shahi paneer"),
 "boondi laddu": ("ASC343", "proxy", "besan ladoo"),
 "chaas": ("ASC022", "proxy", "salted lassi"),
 "rajma chawal": ("ASC165", "proxy", "rajma curry only, rice not included"),
 "set dosa": ("BFP148", "proxy", "plain dosa"),
 "neer dosa": ("BFP148", "proxy", "plain dosa"),
 "grilled sandwich": ("BFP456", "proxy", "tomato sandwich (grilled)"),
}
NONE = ["hara bhara kabab", "navratan korma", "amritsari kulcha", "balushahi", "chicken pizza", "chicken wings", "chole bhature", "dabeli", "falooda",
        "garlic bread", "ghevar", "idiyappam", "jalebi", "masala papad", "misal pav", "modak", "mysore pak",
        "paneer pizza", "pani puri", "paniyaram", "papdi chaat", "patrode", "pepperoni pizza", "pongal",
        "puri bhaji", "sev puri", "solkadhi", "steamed momo", "thali", "thukpa", "vada pav"]
for l in NONE:
    M[l] = ("", "none", "no sensible INDB recipe found by keyword search")
M["hara bhara kabab"] = ("", "none", "only proxy is vegetable seekh kebab ASC377, whose INDB value looks implausible")
M["navratan korma"] = ("", "none", "only proxy is paneer makhana korma ASC225, whose INDB value looks implausible")
# biryani and seekh kebab stay 'pending' until you look at the Khana images (see Step 4)

base = pd.DataFrame([(l, *M.get(l, ("", "pending", ""))) for l in KHANA_CLASSES],
                    columns=["khana_label", "indb_recipe_id", "status", "notes"])
path = DATA / "mapping.csv"
if path.exists():                                   # keep rows you already finished by hand
    old = pd.read_csv(path).fillna("").set_index("khana_label")
    done = old[old.status != "pending"]
    base = base.set_index("khana_label"); base.update(done); base = base.reset_index()
base.to_csv(path, index=False)
print(base.status.value_counts().to_string())

## Step 3: review the mapping (recipe name, serving unit and whether the serving size was guessed)

In [ ]:
mp = pd.read_csv(DATA / "mapping.csv").fillna("")
nut = pd.read_csv(DATA / "nutrition.csv")
rev = mp.merge(nut[["recipe_code", "recipe_name", "serving_unit", "kcal", "nutrition_flag", "serving_guessed"]].fillna({"nutrition_flag": ""}),
               left_on="indb_recipe_id", right_on="recipe_code", how="left")
bad = rev[(rev.status.isin(["match", "variant", "proxy"])) & rev.recipe_name.isna()]
print("mapped ids that do not exist in INDB:", bad.khana_label.tolist())
print("flagged as implausible (kcal/100g > 600 or kcal per unit > 800):", int((rev.nutrition_flag == "check").sum()), "of the mapped classes")
print("kcal = per ONE serving unit (1 idli, 1 bowl...). Rows marked 'check' are likely inflated by frying oil or syrup.\n")
print(rev[rev.status.isin(["match", "variant", "proxy"])][["khana_label", "status", "recipe_name", "serving_unit", "kcal", "nutrition_flag", "serving_guessed"]].to_string(index=False))

dist = DATA / "khana_class_distribution.csv"      # copy this file from /content to Drive if you have it
if dist.exists():
    d = pd.read_csv(dist).merge(mp, left_on="class_name", right_on="khana_label")
    share = d.groupby("status").image_count.sum() / d.image_count.sum()
    print("\nshare of Khana images by mapping status:"); print((share * 100).round(1).to_string())

## Step 4: the two classes that need your eyes
Open ~10 Khana images of **biryani** and **seekh kebab** and decide what they show (chicken/mutton or vegetable), then edit `mapping.csv` on Drive.

In [ ]:
for code in ["ASC122", "ASC123", "BFP142", "BFP199", "ASC377", "OSR153"]:
    r = nut[nut.recipe_code == code].iloc[0]
    print(code, r.recipe_name, "|", r.serving_unit, "| kcal/serving", round(r.kcal))
print("\nIngredient lists for the pizza decision:")
for code in ["ASC376", "BFP122"]:
    print(code, rec[rec.recipe_code == code][["ingredient_name_org", "amount", "unit"]].to_string(index=False), "\n")

def find(pattern, n=15):                        # search INDB recipe names, e.g. find("kebab")
    m = nut[nut.recipe_name.str.lower().str.contains(pattern.lower(), na=False)]
    print(m[["recipe_code", "recipe_name", "serving_unit", "kcal"]].head(n).to_string(index=False))